# SignalForge Week 11 Unsloth ORPO Run-All Notebook

This is the hardened Colab notebook for the current SignalForge repo.

It is designed around the issues we already hit:

- unstable Unsloth installs
- stale repo clones in `/content/SignalForge`
- prompts that were too judge-like and not explicit enough about output format
- weak sample testing after training

What this notebook does:

1. Verifies the Unsloth stack and installs it only if needed.
2. Restarts the runtime once after install.
3. Re-clones the repo fresh from `main`.
4. Exports the train/dev/held-out preference bundle.
5. Loads `Qwen/Qwen2.5-1.5B-Instruct` in 4-bit with Unsloth LoRA.
6. Skips SFT by default, but leaves it available.
7. Runs ORPO on `prompt / chosen / rejected`.
8. Saves the adapter to Google Drive.
9. Prints training history and a dev sanity check.

Important:

- On the very first run in a fresh runtime, the notebook may restart once after installing Unsloth.
- After that restart, just run all cells again from the top.


In [ ]:
import importlib
import os
import subprocess
import sys

def _unsloth_stack_ready():
    modules = ['unsloth', 'unsloth_zoo', 'transformers', 'trl', 'peft', 'datasets']
    try:
        for name in modules:
            importlib.import_module(name)
        return True
    except Exception as exc:
        print(f'Unsloth stack not ready yet: {type(exc).__name__}: {exc}')
        return False

if not _unsloth_stack_ready():
    print('Installing stable Unsloth release for Colab...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', 'setuptools', 'wheel'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', 'unsloth'], check=True)
    print('Restarting runtime now. After restart, run all cells again from the top.')
    os.kill(os.getpid(), 9)
else:
    print('Unsloth stack already available. Continuing.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from datetime import datetime
from pathlib import Path
import json
import shutil
import subprocess

REPO_URL = 'https://github.com/nebiyuephrata/SignalForge.git'
REPO_BRANCH = 'main'
REPO_ROOT = Path('/content/SignalForge')

RUN_STAMP = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
DRIVE_ROOT = Path('/content/drive/MyDrive/SignalForgeRuns')
RUN_DIR = DRIVE_ROOT / f'unsloth_orpo_{RUN_STAMP}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_EXPORT = REPO_ROOT / 'training_data' / 'unsloth' / 'preferences_train.jsonl'
DEV_EXPORT = REPO_ROOT / 'training_data' / 'unsloth' / 'preferences_dev.jsonl'
HELD_OUT_EXPORT = REPO_ROOT / 'training_data' / 'unsloth' / 'preferences_held_out.jsonl'
LOCAL_OUTPUT_DIR = REPO_ROOT / 'training' / 'colab_orpo_outputs'

print({
    'repo_root': str(REPO_ROOT),
    'run_dir': str(RUN_DIR),
})


In [ ]:
if REPO_ROOT.exists():
    print(f'Removing stale repo at {REPO_ROOT}')
    shutil.rmtree(REPO_ROOT)

subprocess.run([
    'git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_ROOT)
], check=True)

print('Fresh repo clone ready.')


In [ ]:
%cd /content/SignalForge
!python training/export_unsloth_datasets.py


In [ ]:
from datasets import Dataset

def read_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = read_jsonl(TRAIN_EXPORT)
dev_rows = read_jsonl(DEV_EXPORT)
held_out_rows = read_jsonl(HELD_OUT_EXPORT)

print({
    'train_rows': len(train_rows),
    'dev_rows': len(dev_rows),
    'held_out_rows': len(held_out_rows),
})
print('Sample task:', train_rows[0]['task_id'], train_rows[0]['dimension'], train_rows[0]['task_type'])
print(train_rows[0]['prompt'][:400])


## Model Setup

Defaults chosen for the first stable run:

- model: `Qwen/Qwen2.5-1.5B-Instruct` pinned to revision `989aa79`
- 4-bit loading
- LoRA rank `16`
- no SFT warm start
- ORPO as the main training step


In [ ]:
import unsloth
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 1536
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
MODEL_REVISION = '989aa79'
RUN_SFT_WARMSTART = True
SFT_MAX_STEPS = 120
ORPO_NUM_EPOCHS = 1
ORPO_LEARNING_RATE = 5e-6

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    revision=MODEL_REVISION,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(type(model))
print({'pad_token': tokenizer.pad_token, 'eos_token': tokenizer.eos_token})


In [ ]:
if RUN_SFT_WARMSTART:
    from trl import SFTConfig, SFTTrainer

    warm_train = Dataset.from_list([{'text': row['sft_text']} for row in train_rows])
    warm_dev = Dataset.from_list([{'text': row['sft_text']} for row in dev_rows])

    sft_trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=warm_train,
        eval_dataset=warm_dev,
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LENGTH,
        args=SFTConfig(
            output_dir=str(LOCAL_OUTPUT_DIR / 'sft_warmstart'),
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            max_steps=SFT_MAX_STEPS,
            learning_rate=1e-4,
            logging_steps=5,
            eval_strategy='steps',
            eval_steps=20,
            save_steps=40,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            report_to='none',
        ),
    )
    sft_trainer.train()
else:
    print('Skipping SFT warm start.')


## ORPO Training

This is the main Path B training step.

We train on:

- `prompt`: the task
- `chosen`: the better response
- `rejected`: the worse response


In [ ]:
from trl import ORPOConfig, ORPOTrainer
import inspect

orpo_train = Dataset.from_list([
    {
        'prompt': row['prompt'],
        'chosen': row['chosen'],
        'rejected': row['rejected'],
        'task_id': row['task_id'],
        'dimension': row['dimension'],
        'task_type': row['task_type'],
    }
    for row in train_rows
])

orpo_dev = Dataset.from_list([
    {
        'prompt': row['prompt'],
        'chosen': row['chosen'],
        'rejected': row['rejected'],
        'task_id': row['task_id'],
        'dimension': row['dimension'],
        'task_type': row['task_type'],
    }
    for row in dev_rows
])

orpo_args = ORPOConfig(
    output_dir=str(LOCAL_OUTPUT_DIR / 'orpo_run'),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=ORPO_LEARNING_RATE,
    num_train_epochs=ORPO_NUM_EPOCHS,
    logging_steps=5,
    save_steps=50,
    eval_strategy='steps',
    eval_steps=25,
    max_length=1536,
    max_prompt_length=1024,
    beta=0.1,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    report_to='none',
    remove_unused_columns=False,
)

trainer_kwargs = {
    'model': model,
    'args': orpo_args,
    'train_dataset': orpo_train,
    'eval_dataset': orpo_dev,
}

sig = inspect.signature(ORPOTrainer.__init__)
if 'processing_class' in sig.parameters:
    trainer_kwargs['processing_class'] = tokenizer
else:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = ORPOTrainer(**trainer_kwargs)
train_result = trainer.train()
print(train_result)


In [ ]:
adapter_dir = RUN_DIR / 'orpo_adapter'
history_path = RUN_DIR / 'trainer_log_history.json'

trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

with open(history_path, 'w') as f:
    json.dump(trainer.state.log_history, f, indent=2)

print({
    'adapter_dir': str(adapter_dir),
    'history_path': str(history_path),
})


In [ ]:
import pandas as pd

history = pd.DataFrame(trainer.state.log_history)
display(history.tail(20))

metric_candidates = [
    'loss',
    'eval_loss',
    'rewards/chosen',
    'rewards/rejected',
    'rewards/accuracies',
    'rewards/margins',
    'logps/chosen',
    'logps/rejected',
]
metric_cols = [c for c in metric_candidates if c in history.columns]
if metric_cols:
    display(history[metric_cols].dropna(how='all').tail(20))
else:
    print('No ORPO metric columns were found in trainer.state.log_history.')


## Dev Sanity Check

This block checks whether the trained model is producing outputs in the expected format.

It does not replace the full benchmark evaluator, but it is a fast way to spot generic drift, over-claiming, and output-shape issues.


In [ ]:
FastLanguageModel.for_inference(model)

def assistant_prompt(row):
    base = row['chosen_chatml'].split('<|im_start|>assistant')[0] + '<|im_start|>assistant\n'
    if row['task_type'] == 'email_grounding':
        return base + 'Subject:'
    return base

def generate_response(row, max_new_tokens=180):
    prompt = assistant_prompt(row)
    inputs = tokenizer([prompt], return_tensors='pt', truncation=True).to('cuda')
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.2,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        repetition_penalty=1.05,
    )
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=False)[0]
    return decoded[len(prompt):].strip() if decoded.startswith(prompt) else decoded.strip()

for i in range(3):
    row = dev_rows[i]
    print(f'\n--- DEV SAMPLE {i} ---')
    print('TASK:', row['task_id'], row['dimension'], row['task_type'])
    print('\nMODEL OUTPUT:\n')
    print(generate_response(row))
    print('\nEXPECTED CHOSEN:\n')
    print(row['chosen'])


In [ ]:
def parse_email_output(text):
    subject = ''
    body_lines = []
    in_body = False
    for line in text.splitlines():
        if line.lower().startswith('subject:'):
            subject = line.split(':', 1)[1].strip()
        elif line.lower().startswith('body:'):
            body_lines = [line.split(':', 1)[1].strip()]
            in_body = True
        elif in_body:
            body_lines.append(line.strip())
    body = '\n'.join([line for line in body_lines if line]).strip()
    return subject, body

def email_rule_check(row, generated_text):
    ground_truth = row['prompt_messages'][1]['content']
    payload = json.loads(ground_truth)
    gt = payload['ground_truth']
    subject, body = parse_email_output(generated_text)
    full = generated_text.lower()
    required_strings = gt.get('required_signal_strings', [])
    banned_phrases = gt.get('banned_phrases', [])

    checks = {
        'has_subject': bool(subject),
        'has_body': bool(body),
        'required_strings_ok': all(s.lower() in full for s in required_strings),
        'banned_phrases_ok': all(p.lower() not in full for p in banned_phrases),
        'question_mark_ok': ('?' in body) if gt.get('require_question_mark') else True,
        'calendar_ok': ('cal.com' in full) if gt.get('require_calendar_link') else ('cal.com' not in full if gt.get('require_calendar_link') is False else True),
        'handoff_ok': (str(gt.get('require_handoff_phrase')).lower() in full) if gt.get('require_handoff_phrase') else True,
    }
    checks['all_ok'] = all(checks.values())
    return checks

email_dev_rows = [row for row in dev_rows if row['task_type'] == 'email_grounding'][:8]
results = []
for row in email_dev_rows:
    generated = generate_response(row)
    checks = email_rule_check(row, generated)
    results.append({'task_id': row['task_id'], 'dimension': row['dimension'], **checks})

sanity_df = pd.DataFrame(results)
display(sanity_df)
print('Email sanity pass rate:', round(float(sanity_df['all_ok'].mean()), 4) if len(sanity_df) else 'n/a')


## Next Step

If the model still produces generic analysis instead of the expected final answer shape:

1. set `RUN_SFT_WARMSTART = True`
2. rerun the notebook
3. compare the new dev sanity outputs against this run

Do not touch `held_out` until the recipe looks stable on `dev`.
